# Phase 2C.0 C3 - Hierarchical Child Index (Colab)

Materialize 512/64 parents with 256/32 children, build one BGE-M3 learned-sparse child index, export the parent corpus and child-to-parent map, validate all references, and save a reusable ZIP to Google Drive. No generator or judge API is called.

## 1. Configuration

In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='7af77304ab317d5a126f9a5ec90397cbcf8cda8e'
RAW_REPO_ID='MatchaMacchiato/newsqa_200_11064_v2.0.0'
RAW_REVISION='b81c8db6847a23272665946c0c43c72e9a212fd9'
ARM_ID='c3_hierarchical'
CHUNKING={'strategy':'hierarchical','chunk_size':512,'chunk_overlap':64,'child_chunk_size':256,'child_chunk_overlap':32}
DRIVE_DIRECTORY='/content/drive/MyDrive/newsqa_phase2c/preparation'
RESTORE_COMPLETED=True
OVERWRITE=False
EXPECTED_ARTICLES=11064
EXPECTED_QUESTIONS=1152
DEVICE='cuda:0'

## 2. Runtime setup

In [ ]:
import hashlib,importlib.metadata as metadata,json,os,pickle,shutil,subprocess,sys,yaml
from datetime import datetime,timezone
from google.colab import drive,userdata
from packaging.version import Version
drive.mount('/content/drive')
ROOT=Path('/content'); PROJECT_ROOT=ROOT/'Text-Mining---NewsQA-RAG'; WORK=ROOT/f'phase2c_{ARM_ID}'
MATERIALIZED=WORK/'materialized'; INDEX_ROOT=WORK/'index'; BUNDLE=WORK/'bundle'; LOGS=WORK/'logs'
DRIVE_ROOT=Path(DRIVE_DIRECTORY); DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
for path in [WORK,BUNDLE,LOGS]: path.mkdir(parents=True,exist_ok=True)
assert not REPO_COMMIT.startswith('SET_TO_'),'Pin REPO_COMMIT after committing these notebooks'
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','FlagEmbedding>=1.4.2'],check=True)
import torch
assert Version(metadata.version('FlagEmbedding'))>=Version('1.4.2')
assert torch.cuda.is_available(),'Enable a Colab GPU runtime'
assert torch.cuda.get_device_capability(0)>=(7,0),'Use T4, L4 or A100; do not use P100'
try: HF_TOKEN=userdata.get('HF_TOKEN') or ''
except Exception: HF_TOKEN=''
os.environ.update({'HF_HOME':str(ROOT/'hf_cache'),'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0','PYTHONPATH':str(PROJECT_ROOT/'common')})
print('GPU:',torch.cuda.get_device_name(0),'| arm:',ARM_ID,'| HF authenticated:',bool(HF_TOKEN))

## 3. Build, export parents, validate, and package

In [ ]:
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def rows(path):
    with open(path,encoding='utf-8') as f: return [json.loads(line) for line in f if line.strip()]
def run(command,label):
    log_path=LOGS/f'{label}.log'; print('$',' '.join(map(str,command)),flush=True)
    with open(log_path,'a',encoding='utf-8') as log:
        process=subprocess.Popen(list(map(str,command)),cwd=PROJECT_ROOT,env={**os.environ,'PYTHONUNBUFFERED':'1'},stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
        code=process.wait()
    if code: raise subprocess.CalledProcessError(code,command)
def validate():
    required=[BUNDLE/'chunks.jsonl',BUNDLE/'testset_resolved.jsonl',BUNDLE/'bge_m3_sparse.pkl',BUNDLE/'config.yaml',BUNDLE/'variant.json',BUNDLE/'parents.jsonl',BUNDLE/'child_parent_map.jsonl']
    assert all(path.exists() for path in required),[str(path) for path in required if not path.exists()]
    chunks=rows(required[0]); questions=rows(required[1]); parents=rows(required[5]); links=rows(required[6])
    ids={x['id'] for x in chunks}; article_ids={x['metadata']['canonical_article_id'] for x in chunks}; parent_ids={x['id'] for x in parents}
    assert len(ids)==len(chunks) and len(article_ids)==EXPECTED_ARTICLES and len(questions)==EXPECTED_QUESTIONS
    assert len(parent_ids)==len(parents) and len(links)==len(chunks)
    assert {x['child_id'] for x in links}==ids and {x['parent_id'] for x in links}<=parent_ids
    assert not ({chunk_id for q in questions for chunk_id in q['relevant_chunk_ids']}-ids)
    with open(required[2],'rb') as f: index=pickle.load(f)
    assert index['size']==len(chunks)
    assert yaml.safe_load(required[3].read_text())['chunking']==CHUNKING
    return {'articles':len(article_ids),'chunks':len(chunks),'parents':len(parents),'questions':len(questions),'index_bytes':required[2].stat().st_size}
archive=DRIVE_ROOT/f'phase2c_{ARM_ID}.zip'
if RESTORE_COMPLETED and archive.exists() and not (BUNDLE/'artifact_manifest.json').exists(): shutil.unpack_archive(archive,BUNDLE)
try:
    statistics=validate(); print('Reusing completed artifact:',statistics)
except Exception as error:
    print('Build required:',type(error).__name__,str(error)[:160])
    if OVERWRITE:
        for path in [MATERIALIZED,INDEX_ROOT,BUNDLE]: shutil.rmtree(path,ignore_errors=True)
        BUNDLE.mkdir(parents=True,exist_ok=True)
    config=yaml.safe_load((PROJECT_ROOT/'configs/config.yaml').read_text()); config['chunking']=CHUNKING
    config_path=WORK/'source_config.yaml'; config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
    if not (MATERIALIZED/'final_deduplicated/chunks.jsonl').exists():
        command=[sys.executable,'-u','scripts/materialize_evaluation_dataset.py','--repo-id',RAW_REPO_ID,'--revision',RAW_REVISION,'--output-root',MATERIALIZED,'--config',config_path,'--db-path',WORK/'temporary_chroma','--skip-vector-index']
        if HF_TOKEN: command += ['--token',HF_TOKEN]
        run(command,'materialize')
    if not (INDEX_ROOT/'index_manifest.json').exists():
        run([sys.executable,'-u','scripts/build_retrieval_models_index.py','--chunks-path',MATERIALIZED/'final_deduplicated/chunks.jsonl','--base-variant-manifest',MATERIALIZED/'manifests/deduplicated.variant.json','--base-config',config_path,'--output-dir',INDEX_ROOT,'--sparse-ids','bge_m3_sparse','--skip-dense','--device',DEVICE],'index')
    copies={MATERIALIZED/'final_deduplicated/chunks.jsonl':'chunks.jsonl',MATERIALIZED/'final_deduplicated/testset_resolved.jsonl':'testset_resolved.jsonl',MATERIALIZED/'manifests/deduplicated.variant.json':'source_variant.json',INDEX_ROOT/'bge_m3_sparse.pkl':'bge_m3_sparse.pkl',INDEX_ROOT/'config_sparse_bge_m3_sparse.yaml':'config.yaml',INDEX_ROOT/'variant_sparse_bge_m3_sparse.json':'variant.json'}
    for source,name in copies.items(): shutil.copy2(source,BUNDLE/name)
    parents={}; links=[]
    with open(BUNDLE/'chunks.jsonl',encoding='utf-8') as handle:
        for line in handle:
            child=json.loads(line); meta=child['metadata']; parent_id=meta['parent_id']
            parent={'id':parent_id,'text':child['parent_text'],'metadata':{'canonical_article_id':meta['canonical_article_id'],'parent_index':meta['parent_index'],'strategy':'hierarchical_parent'}}
            if parent_id in parents: assert parents[parent_id]==parent
            parents[parent_id]=parent; links.append({'child_id':child['id'],'parent_id':parent_id})
    with open(BUNDLE/'parents.jsonl','w',encoding='utf-8') as handle:
        for key in sorted(parents): handle.write(json.dumps(parents[key])+'\n')
    with open(BUNDLE/'child_parent_map.jsonl','w',encoding='utf-8') as handle:
        for row in sorted(links,key=lambda x:x['child_id']): handle.write(json.dumps(row)+'\n')
    statistics=validate()
    artifacts={path.relative_to(BUNDLE).as_posix():{'bytes':path.stat().st_size,'sha256':sha(path)} for path in sorted(BUNDLE.rglob('*')) if path.is_file()}
    manifest={'schema_version':1,'status':'complete','created_at':datetime.now(timezone.utc).isoformat(),'arm_id':ARM_ID,'chunking':CHUNKING,'statistics':statistics,'source':{'repo_id':RAW_REPO_ID,'revision':RAW_REVISION,'repository_commit':REPO_COMMIT},'artifacts':artifacts}
    (BUNDLE/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
    local=Path(shutil.make_archive(str(WORK/f'phase2c_{ARM_ID}'),'zip',root_dir=BUNDLE)); temporary=archive.with_suffix('.zip.tmp'); shutil.copy2(local,temporary); temporary.replace(archive)
assert archive.exists(); print('Complete:',statistics); print('Drive artifact:',archive,f'{archive.stat().st_size/2**20:.1f} MiB'); print('SHA-256:',sha(archive))